# Campus Event Scheduling System
**DTSC5501 — Data Structures and Algorithms | Group Project 1**

**Authors:** Mike Beitner · Luis Echeverry · Christopher Taylor

---

## Project Overview

This project implements a lightweight scheduling system for managing campus events. The system supports adding, searching, sorting, and conflict-checking events efficiently as the event list grows from a handful to thousands. Two independent data structure backends — a dynamic array and a singly linked list — are implemented from scratch and benchmarked against each other across all supported operations.

The project is organized into four parts:

- **Part A** — *Event Storage Structures*: two custom data structure implementations
- **Part B** — *Sorting*: three sorting algorithms benchmarked across both structures
- **Part C** — *Searching & Conflict Detection*: linear/binary search and conflict checking
- **Part D** — *Scalability Challenge*: memory estimation and design at n = 1,000,000

---
## Setup & Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

# CORE DATA STRUCTURES
from event_creator import Event
from dynamic_array import DynamicArrayEvent
from linked_list import EventLinkedList
from converter import converter

# ALGORITHMS
from sorting import insertSort, mergeSort, quickSort
from searching import linear, binary
from conflict import conflict, conflict_naive, conflict_optimized

# TEST DATA & BENCHMARKING
from sample_array import friday, brunch, lunch, hhour, play, stars
from randEvent import genEvents
import benchmark

# ANALYSIS & PLOTTING
import copy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print('All imports successful.')

---
# Part A — Event Storage Structures
## A.1 The Event Class

The core data object is the `Event` class, defined in `event_creator.py`. Each event has five attributes:

| Attribute | Type | Description |
|---|---|---|
| `id` | `int` | Auto-generated unique 6-digit integer |
| `title` | `str` | Name of the event |
| `date` | `str` | Date in `YYYY-MM-DD` format |
| `time` | `str` | Time in `HH:MM` 24-hour format |
| `location` | `str` | Venue or building name |

*Design note - Date and Time formats:* Dates and times are stored as zero-padded strings using the formats YYYY-MM-DD and HH:MM. We chose this representation intentionally so that standard lexicographic string comparison produces the same ordering as chronological time. Since the fields are fixed-width and zero-padded, values such as "2026-03-10" will correctly sort before "2026-11-02", and "09:30" will correctly sort before "14:15". This allows the sorting algorithms to compare the values directly as strings instead of converting them to datetime objects. The main benefit of this approach is simplicity: sorting can be performed with straightforward string comparisons, avoiding additional parsing overhead while still producing correct chronological ordering.

*Design note - Event IDs:* Event IDs are generated using a module-level set called _usedIDs, which records every ID assigned during the current program session. When a new ID is created, it is checked against this set to ensure it has not already been used. If the ID is unique, it is added to the set and assigned to the event. This approach guarantees that IDs remain unique throughout the lifetime of the program. The system also allows IDs to be assigned manually when needed. This is primarily useful for testing and reproducibility—for example, in sample_array.py, where fixed IDs make it easier to construct predictable test cases and verify sorting behavior.

*`sortKey()` method:* The sortKey() method provides a consistent sorting interface across all implemented sorting algorithms. It returns a tuple in the form: (date, time, location)

Python’s tuple comparison evaluates elements from left to right, which naturally enforces the intended priority order:

1. Date – primary sorting criterion

2. Time – secondary ordering within the same date

3. Location – alphabetical tiebreaker when both date and time match

Including location as a final comparison field ensures that sorting remains deterministic even when two events occur at the same date and time.

In [ ]:
# DEMONSTRATING THE EVENT CLASS
print('Sample events from sample_array.py:')
for e in friday:
    print(f'  {e}')

print()
print(f'Sort key for brunch: {brunch.sortKey()}')
print(f'Sort key for stars:  {stars.sortKey()}')
print(f'String comparison works correctly: brunch < stars → {brunch.sortKey() < stars.sortKey()}')

## A.2 Dynamic Array Implementation

The DynamicArrayEvent class (implemented in dynamic_array.py) provides an array-based structure for storing events. Internally, it maintains a fixed-size Python list that acts as the backing array. When the array reaches capacity, the structure automatically resizes by allocating a new array with double the previous capacity and copying the existing elements into it. This resizing strategy is widely used in dynamic array implementations, including Java ArrayList and std::vector. Doubling the capacity keeps the number of resize operations relatively small while allowing the structure to grow as needed. The result is a data structure that provides constant-time indexed access while still supporting dynamic growth.

### Supported Operations & Theoretical Complexities

| Operation | Complexity | Notes |
|---|---|---|
| `append(event)` | O(1) amortized, O(n) worst | O(n) only when resize is triggered; amortized O(1) across many appends |
| `resize()` | O(n) | Copies all elements to a new doubled-capacity array |
| `get_at(index)` | O(1) | Direct index access — the key advantage of arrays |
| `insert(index, event)` | O(n) | Must shift all elements to the right of the insertion point |
| `delete(target_id)` | O(n) | Must scan for the ID, then shift elements left to close the gap |
| `search_by_id(id)` | O(n) | Linear scan through the internal array |
| `sort(algorithm)` | O(n log n) avg | Depends on chosen algorithm; O(n²) worst for insertion sort |
| `list_all()` | O(n) | Copies all live elements into a plain Python list |
| `__len__()` | O(1) | Returns a tracked size counter |

*Why amortized O(1) for append?* Although an individual append() operation can occasionally take O(n) time when a resize occurs, these expensive operations happen relatively infrequently. Because the array doubles in size each time it fills up, the number of resizes grows logarithmically with respect to the number of elements stored. Over the lifetime of many append operations, the total number of element copies remains proportional to O(n). As a result, when the total work is averaged across all append operations, the amortized cost per append is O(1). In practice, this means that most append operations complete in constant time, with only occasional resizing events requiring additional work.

In [ ]:
# DEMONSTRATING DynamicArrayEvent
da = DynamicArrayEvent()
for e in friday:
    da.append(e)

print(f'Length: {len(da)}')
print(f'Event at index 2: {da.get_at(2)}')
print(f'All events: {da.list_all()}')

# SEARCH BY ID
result = da.search_by_id(hhour.id)
print(f'\nSearch for hhour (id={hhour.id}): {result}')

# INSERT AT INDEX
new_event = Event('Coffee Hour', '2026-03-06', '10:00', 'ATLAS', id=99)
da.insert(2, new_event)
print(f'\nAfter inserting Coffee Hour at index 2:')
print(f'  Event at index 2: {da.get_at(2)}')
print(f'  New length: {len(da)}')

# DELETE
da.delete(99)
print(f'\nAfter deleting Coffee Hour: length = {len(da)}')

## A.3 Linked List Implementation

The EventLinkedList class (implemented in linked_list.py) provides a singly linked list implementation for storing events. Each Node in the list contains an Event object along with a pointer (next) that references the following node in the chain. The list maintains a reference to the head node and tracks its current size using a counter. Keeping a size counter allows the program to return the list length in constant time without needing to traverse the structure. This structure differs fundamentally from the dynamic array implementation: instead of storing elements in contiguous memory, each node exists independently and is connected through pointers.

### Supported Operations & Theoretical Complexities

| Operation | Complexity | Notes |
|---|---|---|
| `push(event)` | O(1) | Inserts at the head — no traversal needed |
| `append(event)` | O(n) | Must traverse to the tail to insert at the end |
| `get_at(index)` | O(n) | Must traverse node by node — no random access |
| `insert(index, event)` | O(n) | Must traverse to the insertion point |
| `delete(target_id)` | O(n) | Must traverse to find the target node |
| `search_by_id(id)` | O(n) | Linear traversal from head |
| `sort(algorithm)` | O(n log n) avg | Converts to array, sorts, writes back in place |
| `list_all()` | O(n) | Traverses the full list collecting events |
| `__len__()` | O(1) | Returns tracked size counter |

*Key difference from the array:* Unlike the dynamic array, a linked list does not support constant-time random access. There is no direct way to jump to a particular index in the structure. Operations such as get_at(index) therefore require traversing the list node by node starting from the head, which leads to O(n) time complexity. This limitation also affects sorting. Many sorting algorithms assume efficient indexing or slicing operations, which linked lists do not provide. To address this, the implementation converts the linked list into a standard Python list using converter.py, performs the sort on that list, and then writes the sorted events back into the existing nodes. This approach allows the project to reuse the same sorting algorithms while keeping the linked list structure intact.

*Memory layout difference:* A linked list does not require contiguous memory allocation. Each node can exist anywhere in memory and simply stores a pointer to the next node in the chain. This design eliminates the need for resizing operations, which are required in dynamic arrays when they exceed capacity. However, the tradeoff is additional memory overhead: every node must store an extra pointer (next) along with the event data. In practice, this means linked lists trade memory efficiency and random access speed for structural flexibility and simpler insertion behavior at the head of the list.

In [ ]:
# DEMONSTRATING EventLinkedList
ll = EventLinkedList()
for e in friday:
    ll.append(e)

print(f'Length: {len(ll)}')
print(f'Event at index 2: {ll.get_at(2)}')
print(f'All events: {ll.list_all()}')

# SEARCH BY ID
result = ll.search_by_id(hhour.id)
print(f'\nSearch for hhour (id={hhour.id}): {result}')

# INSERT AT INDEX
new_event = Event('Coffee Hour', '2026-03-06', '10:00', 'ATLAS', id=98)
ll.insert(2, new_event)
print(f'\nAfter inserting Coffee Hour at index 2:')
print(f'  Event at index 2: {ll.get_at(2)}')
print(f'  New length: {len(ll)}')

# DELETE
ll.delete(98)
print(f'\nAfter deleting Coffee Hour: length = {len(ll)}')

## A.4 Array vs Linked List — Structural Comparison

The two data structures used in this project—the dynamic array and the linked list—support the same overall functionality, but their internal designs lead to different performance characteristics. The table below summarizes the theoretical complexity of common operations.

| Operation | Dynamic Array | Linked List | Winner |
|---|---|---|---|
| Random access (`get_at`) | **O(1)** | O(n) | Array |
| Append to end | **O(1)** amortized | O(n) | Array |
| Insert at front | O(n) | **O(1)** (`push`) | Linked List |
| Insert at arbitrary index | O(n) | O(n) | Tie |
| Delete by ID | O(n) | O(n) | Tie |
| Search by ID | O(n) | O(n) | Tie |
| Memory layout | Contiguous | Non-contiguous | — |
| Memory overhead | Low (one array) | Higher (+1 pointer per node) | Array |
| Resize required | Yes (doubles) | No | Linked List |

*Practical Implications* The most significant structural difference between the two approaches is random access. In a dynamic array, elements are stored in contiguous memory, which allows the program to compute the location of any index directly. As a result, operations like get_at(index) run in constant time. A linked list, on the other hand, has no concept of direct indexing. Each element must be reached by traversing the chain of nodes starting from the head, which makes indexed access an O(n) operation. Another key distinction is memory layout. Dynamic arrays store elements in a single contiguous block of memory, which keeps memory overhead low but requires occasional resizing when capacity is exceeded. Linked lists avoid resizing entirely because each node is allocated independently, but they require an additional pointer (next) for every stored element.

*Summary:* For the purposes of this scheduling system, the dynamic array is generally the stronger default choice. Its O(1) random access and lower memory overhead make it well suited for operations like viewing, sorting, and retrieving events by index. The linked list does have one clear advantage: inserting at the front of the structure can be done in O(1) time using push(). However, front insertion is not a dominant operation in this particular application. As a result, the linked list primarily serves a comparative and educational role in this project, illustrating the tradeoffs between contiguous array-based structures and pointer-based data structures.

---
# Part B — Sorting Events

Event sorting is implemented in sorting.py using three classic algorithms: Insertion Sort, Merge Sort, and Quick Sort. Each algorithm accepts an optional key parameter that specifies how events should be compared during sorting. By default, the key function is event.sortKey(), which returns a tuple in the form: (date, time, location). Because Python compares tuples from left to right, this automatically enforces the intended priority: events are sorted by date first, then time, and finally location as a tiebreaker.

## B.1 Algorithm Descriptions

### Insertion Sort

Insertion sort builds the sorted portion of the list incrementally. Starting with the second element, each item is compared against the elements to its left and inserted into the correct position within the already-sorted portion of the list. Elements are shifted right as needed to make room for the insertion.

- **Best case:** O(n) — when the list is already sorted, no shifts are needed
- **Average case:** O(n²) — each element is compared against roughly half the sorted portion
- **Worst case:** O(n²) — when the list is reverse-sorted, every element shifts all the way to the front
- **Space:** O(1) — sorts in place
- **Stable:** Yes

*When it performs well:* Insertion sort tends to perform surprisingly well on small datasets (often fewer than ~50 elements). Its implementation is simple, it has very low constant overhead, and it avoids recursion. It is also especially efficient when the input data is already nearly sorted, since very few shifts are required.

### Merge Sort

Merge sort follows a classic divide-and-conquer strategy. The algorithm repeatedly splits the list into two halves until each sublist contains a single element. These sublists are then merged back together in sorted order. During the merge phase, two sorted lists are combined by repeatedly selecting the smaller front element from either list.

- **Best case:** O(n log n)
- **Average case:** O(n log n)
- **Worst case:** O(n log n) — consistent regardless of input
- **Space:** O(n) — requires auxiliary lists during merging
- **Stable:** Yes

*When it performs well:* Merge sort is the most predictable algorithm in this set. Its O(n log n) runtime holds regardless of how the input data is arranged, making it a reliable option when consistent performance is important. The main tradeoff is its additional memory usage, since temporary arrays are needed during merging.

### Quick Sort

Quick sort is another divide-and-conquer algorithm, but instead of splitting the list directly in half, it selects a pivot element and partitions the list into three groups:

1. elements less than the pivot

2. elements equal to the pivot

3. elements greater than the pivot

The algorithm then recursively sorts the smaller and larger partitions. In this implementation, the pivot is selected randomly, which helps prevent consistently poor pivot choices.

- **Best case:** O(n log n) — pivot consistently splits the list evenly
- **Average case:** O(n log n) — random pivot selection makes worst case very unlikely
- **Worst case:** O(n²) — if the pivot is always the smallest or largest element
- **Space:** O(log n) average — recursive call stack
- **Stable:** No (in this implementation)

*When it performs well:* In practice, quick sort often delivers the fastest real-world performance among comparison-based sorting algorithms. It benefits from strong cache locality and relatively small constant factors. Because this implementation uses random pivot selection, the theoretical O(n²) worst case is extremely unlikely in normal usage.

## B.2 Theoretical Complexity Summary

| Algorithm | Best | Average | Worst | Space | Stable |
|---|---|---|---|---|---|
| Insertion Sort | O(n) | O(n²) | O(n²) | O(1) | Yes |
| Merge Sort | O(n log n) | O(n log n) | O(n log n) | O(n) | Yes |
| Quick Sort | O(n log n) | O(n log n) | O(n²) | O(log n) | No |

In [ ]:
# DEMONSTRATING SORTING ON THE friday SAMPLE DATA
unsorted = [
    stars,   # 22:00
    hhour,   # 16:30
    brunch,  # 09:00
    play,    # 18:30
    lunch,   # 11:30
]

print('Input (unsorted):')
for e in unsorted: print(f'  {e}')

print('\nInsert Sort result:')
for e in insertSort(list(unsorted)): print(f'  {e}')

print('\nMerge Sort result:')
for e in mergeSort(list(unsorted)): print(f'  {e}')

print('\nQuick Sort result:')
for e in quickSort(list(unsorted)): print(f'  {e}')

## B.3 Benchmark Results

To evaluate the relative performance of the sorting algorithms, runtime benchmarks were collected across all three algorithms on both data structures. Tests were performed using randomly generated event datasets of size n = 50, 500, 5,000, and 50,000. For each dataset size, every algorithm was executed three separate times, and the runtimes were averaged. Repeating the trials helps smooth out small timing fluctuations caused by factors such as background processes or normal system variability. To ensure a fair comparison, each trial begins with a fresh deep copy of the original input dataset. This guarantees that no algorithm receives an already-sorted list as input, which could otherwise bias the results—especially for algorithms like insertion sort that perform much faster on nearly sorted data. This setup allows the benchmark to reflect the algorithms’ performance under consistent and comparable conditions across both the dynamic array and linked list implementations.

In [ ]:
print('Running sorting benchmarks — this may take a minute...')
sort_results = benchmark.benchmark_sorting()
print('\nSorting benchmarks complete.')

In [ ]:
# PLOT SORTING RESULTS
sizes = [50, 500, 5000, 50000]
alg_colors = {'Insertion': 'royalblue', 'Merge': 'darkorange', 'Quick': 'green'}
structures = ['Array', 'Linked List']

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
fig.suptitle('Sorting Algorithm Runtime Comparison', fontsize=14, fontweight='bold')

for ax, structure in zip(axes, structures):
    for alg, color in alg_colors.items():
        times = [sort_results[alg][structure][n] for n in sizes]
        ax.plot(sizes, times, marker='o', label=alg, color=color)
    ax.set_title(structure)
    ax.set_xlabel('n (number of events)')
    ax.set_ylabel('Average runtime (seconds)')
    ax.legend()
    ax.set_xscale('log')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sorting_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to sorting_results.png')

![*Figure 1.* Sort algorithm performance test results.](sort_results.png)


### **Figure 1.** Sort algorithm performance test results.

## B.4 Analysis

**Expected observations:**

Before running the experiments, it is helpful to outline the expected behavior based on algorithmic theory. These predictions provide a baseline for interpreting the measured runtimes.

- For very small datasets (around n = 50), insertion sort may perform competitively with—or even slightly faster than—merge sort and quick sort. Its implementation has minimal overhead and does not rely on recursion, which can give it an advantage on small inputs.

- As the dataset size increases, insertion sort’s O(n²) average-case complexity should begin to dominate. Once the input grows into the thousands of elements, its runtime is expected to increase much more rapidly than the other algorithms. By n = 50,000, it will likely be significantly slower than both merge sort and quick sort.

- Both merge sort and quick sort have O(n log n) average complexity, so their runtime curves should grow much more slowly as the input size increases. When plotted, these algorithms typically show near-linear growth on a logarithmic scale.

- In many practical implementations, quick sort tends to run slightly faster than merge sort. This is often attributed to better cache locality and smaller constant factors in the inner loops. However, merge sort usually provides more consistent performance, since its O(n log n) runtime does not depend on pivot quality.

- The linked list implementation may show slightly slower overall runtimes compared to the dynamic array. In this project, sorting a linked list involves converting the list into a standard Python list, performing the sort, and then writing the sorted results back into the existing nodes. The extra conversion step introduces additional overhead that does not occur in the array-based implementation.

B.4 Sorting Analysis

From our benchmark results we have determined that there is indeed a significant difference in runtime when comparing our different sorting algorithms. This follows our theoretical observations as the insertion sort performed the worst of all three sorting algorithms, which we predicted. This was largely due to the fact that the dataset size increase resulted in significantly worse performance due to its time complexity being $O(n^2)$. Although we theorized it to perform a bit faster when having a lower dataset size of N = 50, it still performed slower than the other sorting algorithms, running at 0.000159 seconds for the dynamic array and 0.000434 seconds for the linked list. However, at N = 50,000 our theoretical observation was correct that insertion sort would perform the worse, as its performance was a staggering 245.254482 seconds for the dynamic array and 458.777888 seconds for the linked list. We also initially believed that quick sort would perform faster than merge sort in our theoretical observations, however from our benchmark results merge sort was the best performer. At N = 50,000, quick sort completed in 0.697710 seconds for the dynamic array and 0.787503 seconds for the linked list. These performance times were slower than merge sort as at N = 50,000, the performance speeds were completed in 0.458003 seconds for the dynamic array and 0.4227471 seconds for the linked list. Since the merge sort algorithm is more consistent in its performance it could explain why its speed was faster in this instance, as the quick sort is able to reach its worse case time complexity of $O(n^2)$ when it chooses a bad pivot. As for the performance comparison of the linked list and dynamic arrays, our theoretical prediction that the linked list would perform slower than the dynamic was backed up from our benchmark results, especially when using insertion sort. Overall our benchmark results supports our conclusion that implementing $O(nlogn)$ sorting algorithms like merge and quick sort on dynamic arrays would be essential for handling large data sets more effectively.


---
### Results


*Dynamic Array*

| N | Insertion Sort | Merge Sort | Quick Sort |
|---|---|---|---|
| 50 | 0.000159 seconds | 9.846667e-05 seconds | 0.000133 seconds |
| 500 | 0.016448 seconds|0.001654 seconds | 0.002192 seconds | 
| 5000 | 1.584436 seconds | 0.019559 seconds | 0.027317 seconds |
| 50000 |245.254482 seconds  | 0.458003 seconds | 0.697710 seconds | 



*Linked List*

| N | Insertion Sort | Merge Sort | Quick Sort |
|---|---|---|---|
| 50 | 0.000434 seconds | 9.720000e-05 seconds | 0.000137 seconds |
| 500 | 0.015448 seconds|0.001579 seconds | 0.002020 seconds | 
| 5000 | 1.596537 seconds |0.019411 seconds | 0.027627 seconds |
| 50000 |458.777888 seconds  | 0.4227471seconds | 0.787503 seconds | 

---
# Part C — Searching & Conflict Detection
## C.1 Searching

Two search algorithms are implemented in searching.py: linear search and binary search. Both functions are designed to work with either a standard Python list or an EventLinkedList. If a linked list is provided, it is first converted into a plain list using converter.py so that the search algorithms can operate on a uniform structure. Each search function returns a tuple in the form: (index, iterations)

- index — the position of the found element (or -1 if the element is not present)

- iterations — the number of loop iterations required to complete the search

Tracking the number of iterations provides a simple way to observe how the algorithms scale as the dataset grows.

### Linear Search — O(n)

Linear search works by scanning the dataset from left to right, comparing each event’s ID to the target value until a match is found or the list is exhausted. This approach requires no pre-processing or sorting, which makes it simple and flexible, but also means the algorithm may need to examine many elements before finding the target.

- **Best case:** O(1) — target is the first element
- **Average case:** O(n/2) → O(n) — target is somewhere in the middle
- **Worst case:** O(n) — target is the last element or not present
- **Requires sorted input:** No

Because the algorithm simply walks through the data sequentially, its runtime grows linearly with the size of the dataset.

### Binary Search — O(log n)

Binary search takes a different approach. Instead of scanning sequentially, it repeatedly cuts the search range in half. At each step, the algorithm compares the target value to the middle element of the current search range:

- If the target matches the middle element, the search ends.

- If the target is smaller, the search continues in the left half.

- If the target is larger, the search continues in the right half.

By eliminating half of the remaining elements at each step, binary search reduces the number of comparisons dramatically.

- **Best case:** O(1) — target is the middle element on the first check
- **Average case:** O(log n)
- **Worst case:** O(log n)
- **Requires sorted input:** Yes — the list must be sorted by the same key passed to `binary()`

Since the search range shrinks exponentially, binary search scales extremely well as the dataset grows.

### Complexity Comparison

| Algorithm | Best | Average | Worst | Requires Sorted Input |
|---|---|---|---|---|
| Linear Search | O(1) | O(n) | O(n) | No |
| Binary Search | O(1) | O(log n) | O(log n) | Yes |

*Practical Tradeoff:* Binary search is dramatically faster on large datasets, but it only works when the input data is already sorted. For example, with a dataset of 50,000 events, binary search would require at most about 16 comparisons, since log2(50,000) ≈ 15.6 . By contrast, a linear search might need to check every element, potentially requiring up to 50,000 comparisons. However, sorting the data also has a cost. If the dataset is initially unsorted and we only need to perform a single search, it is usually faster to run a linear scan. Sorting first (which costs O(n log n)) and then performing a binary search (O(log n)) would take longer overall than simply scanning the list once (O(n)). Binary search becomes worthwhile when the data can remain sorted and reused across many searches, allowing the cost of sorting to be amortized over multiple queries.

In [ ]:
# DEMONSTRATING LINEAR AND BINARY SEARCH
sorted_friday = quickSort(list(friday), key=lambda e: e.id)
target = hhour.id

print(f'Searching for event with id = {target} ({hhour.title})')
print(f'List size: {len(friday)} events')
print()

idx, iters = linear(target, list(friday))
print(f'Linear search: found at index {idx} in {iters} iteration(s)')

idx, iters = binary(target, sorted_friday, key=lambda e: e.id)
print(f'Binary search: found at index {idx} in {iters} iteration(s)')

## C.2 Search Benchmark Results

To evaluate performance, search runtimes were measured for datasets of n = 50, 500, 5,000, and 50,000 events. The target for each search was always set to the last element in the sorted list, which represents the worst-case scenario for linear search. This ensures the comparison between linear and binary search is as fair as possible, since linear search must scan every preceding element before finding the target. Additionally, linear search was tested on both unsorted and sorted data. This confirms a key point: sorting the input has no effect on linear search performance, since linear search does not take advantage of ordering. This benchmark design allows a clear demonstration of how linear and binary search scale with increasing dataset size, as well as the impact of input ordering on search efficiency.

In [ ]:
print('Running searching benchmarks...')
search_results = benchmark.benchmark_searching()
print('\nSearching benchmarks complete.')

In [ ]:
# PLOT SEARCHING RESULTS
sizes = [50, 500, 5000, 50000]

fig, ax = plt.subplots(figsize=(9, 5))
ax.set_title('Search Algorithm Runtime Comparison', fontsize=13, fontweight='bold')

ax.plot(sizes, [search_results['linear']['unsorted'][n] for n in sizes],
        marker='o', label='Linear (unsorted)', color='royalblue')
ax.plot(sizes, [search_results['linear']['sorted'][n] for n in sizes],
        marker='s', label='Linear (sorted)', color='cornflowerblue', linestyle='--')
ax.plot(sizes, [search_results['binary']['sorted'][n] for n in sizes],
        marker='^', label='Binary (sorted)', color='darkorange')

ax.set_xlabel('n (number of events)')
ax.set_ylabel('Average runtime (seconds)')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('searching_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to searching_results.png')

![*Figure 2.* Search algorithm performance test results.](search_results.png)


### **Figure 2.** Search algorithm performance test results.

## C.3 Search Analysis


**Expected observations:**

Based on theoretical expectations, we can anticipate the following trends:
- Linear search should perform almost identically on sorted and unsorted data. Since it examines each element sequentially, pre-sorting the list does not improve its performance.
- Binary search should require dramatically fewer iterations as dataset size increases. Its runtime grows logarithmically, while linear search grows linearly, so the difference becomes especially noticeable at large n.
- For very small datasets (n ≈ 50), the difference between linear and binary search will likely be minimal. In fact, the extra bookkeeping involved in binary search—calculating midpoints and adjusting search bounds—may even make it slightly slower than a simple linear scan.


C.3 Search Analysis

As expected, the runtime for linear search was consistent regardless of whether the dataset was sorted or unsorted. Because the algorithm must examine elements sequentially to find the target, the inherent ordering of the data provided no performance benefit, as reflected by the overlapping trajectories of the two linear search curves in our benchmarks. Binary search demonstrated significant performance advantages as the dataset size increased. While the linear search runtime grew at a visible rate as n scaled toward 50,000, the binary search runtime remained nearly constant, appearing as a flat line on our logarithmic scale. While we anticipated that binary search might be slightly slower for very small datasets (n≈50) due to the overhead of midpoint calculations and boundary adjustments, our data shows that binary search was superior even at our smallest sample size. The cost of sequential iteration in linear search becomes a bottleneck so rapidly that binary search is the more efficient choice even for minimal datasets. In summary, binary search is the clear winner for large-scale event management.

---
#### Results



*Searching Algorithms*


| N | Unsorted Linear Search |Sorted Linear Search | Binary Search |
|---|---|---|---|
| 50 | 0.000179 seconds | 0.000339 seconds | 0.000262 seconds |
| 500 | 0.000200 seconds|0.000257 seconds | 0.000222 seconds | 
| 5000 | 0.000686 seconds | 0.000972 seconds | 0.000241 seconds |
| 50000 |0.004487 seconds  | 0.010381 seconds | 0.000250 seconds | 


## C.4 Conflict Detection

In this system, two events are considered to be in conflict if they occur at the same date, time, and location. The project provides two implementations for detecting conflicts, both located in conflict.py.

### Naive Conflict Detection — O(n²)

The simplest approach is a nested loop that compares every pair of events: 

```
for i in range(n):
    for j in range(i+1, n):
        if same location, date, and time → conflict
```

For each event i, the algorithm checks every event j > i to see if all three fields match. While this method is correct and straightforward, it is also very slow: with 10,000 events, it performs roughly 50 million comparisons. The runtime grows quadratically with dataset size, making it impractical for large event lists.

### Optimized Conflict Detection — O(n log n)

The optimized approach takes advantage of sorting. Events are first sorted by their sortKey(), which is the (date, time, location) tuple. Once sorted, any conflicting events are guaranteed to be adjacent in the list, so a single linear scan is enough to find all conflicts:

```
sorted_events = sort_func(events, key=sortKey)
for i in range(len(sorted_events) - 1):
    if sorted_events[i].sortKey() == sorted_events[i+1].sortKey() → conflict
```

The total complexity is:

- $O(n \log n)$ for the sort

- $O(n)$ for the linear pass

The runtime is dominated by the sort. This method is much faster than the naive approach for large datasets.

*Why this works*: Sorting ensures that events with identical (date, time, location) tuples are placed next to each other. Once sorted, it is sufficient to check only consecutive events, rather than every pair in the dataset.

### Complexity Comparison

| Approach | Time Complexity | Space Complexity | Notes |
|---|---|---|---|
| Naive | O(n²) | O(1) | No sorting required |
| Optimized (insertSort) | O(n²) | O(1) | Sort dominates |
| Optimized (mergeSort) | O(n log n) | O(n) | Best consistent performance |
| Optimized (quickSort) | O(n log n) avg | O(log n) | Fastest in practice |

This comparison highlights how sorting first can transform a quadratic-time problem into one that scales efficiently with dataset size, particularly when using merge sort or quick sort.

In [ ]:
# DEMONSTRATING CONFLICT DETECTION
from dynamic_array import DynamicArrayEvent

mock = DynamicArrayEvent()
for e in friday:
    mock.append(e)

# ADD TWO CONFLICTING EVENTS
c1 = Event('Rap Battle Tryouts', '2026-11-25', '09:00', 'Folsom Field')
c2 = Event('Rival Football Game', '2026-11-25', '09:00', 'Folsom Field')
mock.append(c1)
mock.append(c2)

print(f'Total events in schedule: {len(mock)}')
print()

# NAIVE DETECTION
conflicts_naive = conflict_naive(mock)
print(f'Naive detection found {len(conflicts_naive)} conflict(s):')
for pair in conflicts_naive:
    e1, e2 = pair
    print(f'  "{e1.title}" conflicts with "{e2.title}" at {e1.location}, {e1.date}, {e1.time}')

print()

# OPTIMIZED DETECTION
conflicts_opt = conflict_optimized(mock, mergeSort)
print(f'Optimized detection (mergeSort) found {len(conflicts_opt)} conflict(s):')
for e1, e2 in conflicts_opt:
    print(f'  "{e1.title}" conflicts with "{e2.title}" at {e1.location}, {e1.date}, {e1.time}')

## C.5 Conflict Detection Benchmark Results

To evaluate performance, conflict detection runtimes were measured on datasets of n = 100, 500, 5,000, and 10,000 events. For the naive $O(n^2)$ approach, measurements were only taken up to n = 5,000, since beyond that the quadratic growth makes the runtime prohibitively long. The optimized $O(n \log n)$ methods were benchmarked across all dataset sizes. This setup allows a clear comparison of how much the sorting-based approach improves performance, particularly as the number of events grows. By testing at these scales, the benchmarks highlight the practical benefits of leveraging sort-and-scan for conflict detection over the simple nested-loop approach.

In [ ]:
print('Running conflict detection benchmarks...')
conflict_results = benchmark.benchmark_conflict()
print('\nConflict benchmarks complete.')

In [ ]:
# PLOT CONFLICT DETECTION RESULTS (ARRAY STRUCTURE)
conf_sizes = [100, 500, 5000, 10000]
alg_colors = {
    'Naive':      'red',
    'insertSort': 'royalblue',
    'mergeSort':  'darkorange',
    'quickSort':  'green'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
fig.suptitle('Conflict Detection Runtime Comparison', fontsize=14, fontweight='bold')

for ax, structure in zip(axes, ['Array', 'LinkedList']):
    for alg, color in alg_colors.items():
        available = [n for n in conf_sizes if n in conflict_results[structure][alg]]
        times = [conflict_results[structure][alg][n] for n in available]
        ax.plot(available, times, marker='o', label=alg, color=color)
    ax.set_title(structure)
    ax.set_xlabel('n (number of events)')
    ax.set_ylabel('Average runtime (seconds)')
    ax.legend()
    ax.set_xscale('log')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('conflict_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to conflict_results.png')

![*Figure 3.* Conflict detection performance test results.](conflict_results.png)


### **Figure 3.** Conflict detection performance test results.

## C.6 Conflict Detection Analysis


**Expected observations:**

Based on theoretical expectations, the following trends are anticipated:

- The naive $O(n^2)$ conflict detection method should experience rapidly increasing runtimes as dataset size grows. By n = 10,000, it is expected to become practically unworkable due to the sheer number of comparisons required.

- Optimized detection using merge sort or quick sort should scale much more gracefully, producing clean $O(n \log n)$ curves. These approaches leverage sorting to reduce the number of comparisons to a single linear pass after ordering.

- Using insertion sort for the optimized approach is expected to resemble the naive method for large datasets. Since insertion sort itself is $O(n^2)$, its sorting step dominates, and overall performance will degrade similarly as n increases.

- When comparing array and linked list backends, both should show similar scaling trends. The linked list is likely to be slightly slower because converting it to a plain Python list via converter.py introduces additional overhead before sorting and scanning.


C.6 Conflict Detection Analysis

The empirical benchmark results align closely with our theoretical expectations, demonstrating the critical impact of algorithm selection and structure choice on performance. As predicted, the naive $O(n^2)$ approach became prohibitively slow as n approached 10,000, creating a steep performance cliff. In contrast, the optimized approaches using MergeSort and QuickSort maintained high efficiency by transforming the conflict detection problem into a sorting-plus-linear-scan process, yielding the expected $O(n \log n)$ time complexity. The benchmarks highlight the dominance of the sorting step. When InsertionSort was used for the optimized approach, the overall performance was significantly hampered by its $O(n^2)$ complexity, rendering it nearly as slow as the naive approach for larger datasets. MergeSort and QuickSort outperformed InsertionSort in improving conflict detection efficiency across all n. Both array and linkedlist event structures followed near identical runtime trajectories across all three tests. This data supports the conclusion that a naive nested-loop conflict detection on unsorted data will not work regardless of structure.

---
### Conflict Results


*Dynamic Arrays*

| N | Naive | Insertion Sort | Merge Sort | Quick Sort|
|---|---|---|---| ---|
|100|0.000273 seconds|0.001259 seconds|0.000501 seconds|0.000603 seconds|
|500|0.009020 seconds|0.024550 seconds|0.002552 seconds|0.003960 seconds|
|5000|0.875134 seconds|2.981901 seconds|0.035535 seconds|0.050319 seconds|
|10000|**N/A**|15.428314 seconds|0.088872 seconds|0.135598 seconds|



*Linked Lists*

| N | Naive | Insertion Sort | Merge Sort | Quick Sort|
|---|---|---|---| ---|
|100|0.000299 seconds|0.001201 seconds|0.000436 seconds|0.000626 seconds|
|500|0.009040 seconds|0.025884 seconds|0.002924 seconds|0.003825 seconds|
|5000|0.923255 seconds|3.000228 seconds|0.035358 seconds|0.052843 seconds|
|10000|**N/A**|14.444262 seconds|0.090099 seconds|0.139516 seconds|



---
# Part D — Scalability Challenge
## D.1 Memory Estimation at n = 1,000,000 Events

When thinking about scaling to a million events, it’s useful to estimate the memory footprint of each data structure and the event objects themselves.

### Per-Event Memory Cost

Each Event object stores five attributes. Approximate memory usage per attribute is as follows:

| Attribute | Type | Approximate Size |
|---|---|---|
| `id` | Python `int` | ~28 bytes |
| `title` | Python `str` | ~50–70 bytes (varies by length) |
| `date` | Python `str` | ~59 bytes (`YYYY-MM-DD` = 10 chars) |
| `time` | Python `str` | ~54 bytes (`HH:MM` = 5 chars) |
| `location` | Python `str` | ~60–80 bytes (varies by length) |
| Object overhead | Python object base | ~56 bytes |

**Estimated per-event cost: ~300–400 bytes** For 1,000,000 events, that adds up to roughly 300–400 MB just for the event objects themselves, independent of whether they are stored in an array or linked list.

### Structure Overhead

**Dynamic Array:**
The dynamic array stores references (pointers) to objects, not the objects themselves. On a 64-bit system, each reference is 8 bytes.

- At 1,000,000 events, the array of references consumes ~8 MB.

- Since the array doubles in size when full, the maximum reserved capacity could be up to 2× the current size, giving a worst-case overhead of ~16 MB.

Total overhead: ~8–16 MB in addition to the event objects.

**Linked List:**
Each node wraps a reference to an event plus a next pointer. Memory calculation per node:

- Python object base: ~56 bytes

- Event reference: 8 bytes

- next pointer: 8 bytes

Total per-node overhead: ~72 bytes. At 1,000,000 nodes, that adds up to ~72 MB — roughly 5–9× more overhead than the dynamic array.

| Structure | Reference/Node Overhead at n=1M |
|---|---|
| Dynamic Array | ~8–16 MB |
| Linked List | ~72 MB |

**Conclusion:** At large scale, the dynamic array is far more memory-efficient. The linked list’s per-node overhead accumulates quickly, adding nearly an order of magnitude more structural overhead than the array when storing a million events. This analysis reinforces why arrays are generally preferred for large datasets where random access and memory efficiency are important.

In [ ]:
import sys

# MEASURE ACTUAL PYTHON OBJECT SIZES
sample_event = Event('Campus Concert', '2026-06-15', '19:00', 'Macky Auditorium')

print('Actual Python object sizes (sys.getsizeof — direct size only, not referenced objects):')
print(f'  Event object base:  {sys.getsizeof(sample_event)} bytes')
print(f'  id (int):           {sys.getsizeof(sample_event.id)} bytes')
print(f'  title (str):        {sys.getsizeof(sample_event.title)} bytes')
print(f'  date (str):         {sys.getsizeof(sample_event.date)} bytes')
print(f'  time (str):         {sys.getsizeof(sample_event.time)} bytes')
print(f'  location (str):     {sys.getsizeof(sample_event.location)} bytes')

from linked_list import Node
sample_node = Node(sample_event)
print(f'\n  Node object base:   {sys.getsizeof(sample_node)} bytes')

# PROJECTION TO n = 1,000,000
n = 1_000_000
event_size = (sys.getsizeof(sample_event) +
              sys.getsizeof(sample_event.id) +
              sys.getsizeof(sample_event.title) +
              sys.getsizeof(sample_event.date) +
              sys.getsizeof(sample_event.time) +
              sys.getsizeof(sample_event.location))

array_overhead  = 8 * n          # 8 bytes per reference
ll_overhead     = sys.getsizeof(sample_node) * n

print(f'\nProjected memory usage at n = {n:,}:')
print(f'  Event objects (both structures): ~{event_size * n / 1e6:.1f} MB')
print(f'  Array overhead:                  ~{array_overhead / 1e6:.1f} MB')
print(f'  Linked list node overhead:       ~{ll_overhead / 1e6:.1f} MB')
print(f'  Array total:                     ~{(event_size * n + array_overhead) / 1e6:.1f} MB')
print(f'  Linked list total:               ~{(event_size * n + ll_overhead) / 1e6:.1f} MB')

## D.2 HPC Scalability Benchmark


To handle large-scale data past the limits of local development on personal computers, we transitioned to the [UC Denver Alderaan High-Performance Computing cluster](https://ccm-docs.readthedocs.io/en/latest/alderaan/). The entire repository is cloned there, and a bash wrapper script to queue the scalability-challenge.py module was created. The benchmark script evaluates three core operational phases:


1. Data Generation & Loading: Measuring the overhead of initializing 1,000,000 event objects and populating the two distinct data structures.


2. Sorting Performance: Executing a MergeSort on both structures to compare how memory layout influences sorting speed. MergeSort was the highest-performing sort algorithm on smaller datasets, and is therefore most suitable for stress-testing.


3. Search Performance: Performing a binarySearch for a randomized target ID to measure the efficiency of random access versus sequential traversal. Linear search was not attempted.


**Expected Observations**
Based on benchmarking outcomes from smaller datasets, we expect the following trends

-  The EventLinkedList will likely experience higher latency than the DynamicArrayEvent in both MergeSort and BinarySearch. This is due to a lack of contiguous memory allocation to speed up data traversal.

- As n=1,000,000, the memory management overhead for the linked list—storing extra pointers for every single node—will be the primary factor in its performance degradation compared to the more compact dynamic array.


**Implementation**

The scalability workload was executed on the UC Denver Alderaan HPC cluster. To ensure accurate measurement, the Python benchmark used time.perf_counter() to isolate execution time for each stage of the stress test. For example, MergeSort runtime was calculated with:


```
# Measuring MergeSort latency
start = time.perf_counter()
arr.sort()
end = time.perf_counter()
merge_array = end - start
```

The job was scheduled via a SLURM scheduler bash script configured with sufficient memory:


```
#SBATCH --cpus-per-task=16
#SBATCH --mem=64G
#SBATCH --time=2-00:00:00
```

This script requests 64GB of memory to accomodate the memory overhead referenced above. By requesting a dedicated node, we minimized the risk of interference from other scheduled tasks. The script also activates a custom conda environment containing numpy and matplotlib:

| Package	| Version |
| :--- | :--- |
| python	| 3.12.11 |
| numpy	| 2.4.2 |
| matplotlib	| 3.10.8 |

Output is saved to ```alderaan-hpc/scalability-results.txt```, preserved in the repository.

## D.3 Suggested Optimizations

Handling 1,000,000 events efficiently requires rethinking several aspects of the current design:

### 1. Indexing

Right now, search_by_id() performs a linear scan (O(n)), which becomes impractical at a million events. Introducing a hash map index (dict in Python) mapping id → Event would allow O(1) average-case lookups at the cost of roughly 8–16 MB of additional memory for the index. Secondary indexes could also be added for fields like date or location. For example, storing events in sorted arrays by date allows binary search, giving O(log n) range queries.

### 2. Hybrid Structure

A skip list or B-tree could give O(log n) insert, delete, and search simultaneously, while keeping events sorted without full re-sorting. This is particularly useful if inserts and deletes are frequent. For read-heavy workloads (many searches, few writes), keeping a sorted dynamic array and using binary search is already close to optimal.

### 3. External/Disk Storage

One million events at ~400 bytes each is ~400 MB of RAM. For systems that cannot hold all events in memory, a database-backed approach (e.g., SQLite) with indexed columns for date, time, and location allows O(log n) queries directly on disk. This avoids loading all events into memory while still enabling fast searches and conflict detection.

### 4. Lazy Sorting

Rather than sorting the entire list every time a sort is requested, maintain a dirty flag that tracks whether the data has changed. Only re-sort when the flag is set (after insertions or deletions). For workloads with many reads between writes, this avoids repeated, unnecessary sorting.

## D.3 Parallel Conflict Detection Sketch

The current conflict detection algorithm is sequential, but it can be parallelized across multiple CPU cores using a partition-then-merge strategy:
1. Sort all n events by sortKey() once — O(n log n), single-threaded
2. Divide the sorted list into k equal partitions (one per CPU core)
3. Each core independently scans its partition for adjacent conflicts — O(n/k) per core
4. After each core finishes, check the boundary between adjacent partitions
   (the last event of partition i and first event of partition i+1)
   — O(k) boundary checks on the main thread
5. Merge all local conflict lists — O(total conflicts found)

*Why the boundary check matters:* Conflicts that span two partitions would be missed if each core only looks at its own slice. Checking the boundaries ensures these edge cases are caught.

*Theoretical speedup:* The scanning phase reduces from O(n) to O(n/k) with k cores. The sort remains single-threaded here, but it could also be parallelized (parallel merge sort). For example, with 8 cores and 1,000,000 events, the scan phase could be roughly 8× faster.

*Python implementation note:* Python’s Global Interpreter Lock (GIL) prevents true CPU parallelism with threads. To achieve parallel performance, one could use multiprocessing (separate processes) or implement the scan in a lower-level language like C or C++ for true multi-core execution.

In [ ]:
# SCALABILITY SMOKE TEST — GENERATING AND SORTING n=1,000,000 EVENTS
# WARNING: THIS CELL WILL TAKE SEVERAL MINUTES AND USE SIGNIFICANT MEMORY
# UNCOMMENT TO RUN

# import time
# print('Generating 1,000,000 random events...')
# start = time.perf_counter()
# big_list = genEvents(1_000_000)
# gen_time = time.perf_counter() - start
# print(f'  Generated in {gen_time:.2f}s')
#
# print('Sorting with mergeSort...')
# start = time.perf_counter()
# sorted_big = mergeSort(big_list)
# sort_time = time.perf_counter() - start
# print(f'  Sorted in {sort_time:.2f}s')
#
# print(f'First event: {sorted_big[0]}')
# print(f'Last event:  {sorted_big[-1]}')

print('Scalability smoke test cell (commented out by default to avoid long runtimes).')
print('Uncomment the code above to run the full 1,000,000 event test.')

---
# Summary & Conclusions

*(To be completed after all benchmarks are run and graphs are reviewed.)*

Here’s what this section should address once the data is available:

- Sorting performance: Identify which algorithm consistently performed best in practice. Note any crossover points where one algorithm overtakes another as dataset size increases.

From our benchmark tests, it was found that the best performing sorting algorithm was the merge sort, which outperformed both the quick sort and insertion sort algorithms. Although our theoretical observations had the insertion sort outperform merge and quick sort, this wasn't the case in our benchmark results, as both merge and quick sort performed faster than insertion. Although our plot didn't show any crossover point, we did see that after N = 50 the performance times significantly changed for our algorithms. For example the performance time for insertion sort drastically increased in time as the N size grew, demonstrating how much an increase in N size negatively affects performance time. Meanwhile, the merge sort plot maintained the fastest performance over each increase size in N, showing yet again that it was the best performing algorithm in practice. 

- Array vs. linked list: Compare the two backends for sorting. Which structure was faster, and how significant was the difference?

When comparing the sorting performance of our data structure, the dynamic array proved to be the fastest data structure compared to our linked list. This performance difference can be explicitly demonstrated when comparing our performance time at N = 50,000 when using our insertion sort algorithm. At this size of N, the dynamic array took 245.254 seconds, compared to the performance of the linked list, which took 458.777 seconds. This difference in performance time is over 213 seconds, showing that the dynamic array data structure is more efficient for our sorting algorithm benchmark tests. 

- Search efficiency: Highlight how linear and binary search differed at large n, showing the dramatic reduction in iterations and runtime for binary search on sorted datasets.

After running our benchmark tests for our search algorithm, linear and binary search we found that our results coincided with our theoretical observations, as the binary search performed significantly faster as our N size increased. However, the one caveat to this performance difference, is that generally the binary search requires sorted datasets. In our large dataset, the performance difference at N = 50,000 took 0.000250 seconds for our binary search, compared to the 0.010381 seconds our linear search took. However, the unsorted linear search performed better that the sorted linear search as it took 0.004487 seconds. These results demonstrate that the most efficient search algorithm at large datasets is indeed the binary search. 


- Conflict detection limits: Determine the dataset size at which the naive O(n²) approach becomes impractical compared to optimized O(n log n) methods.

- Practical implications: Summarize what these results suggest about choosing algorithms and data structures for real-world scheduling systems. Consider trade-offs between memory, runtime, and operation frequency (e.g., reads vs. writes).

Once the benchmarks are complete, this section will tie together the theoretical expectations with the empirical results, providing clear guidance on design decisions for large-scale event management systems.